# Outline

Will be using *Iterative or KNN Imputer* to fill the missing values

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
houses = pd.read_csv('C:/Users/Wajih/Work (DataScience)/CAPSTONE PROJECTS/Real Estate Project/Zameen.com/Outlier Detection/houses_outlier_handled_new.csv',index_col=0)

In [3]:
houses.columns

Index(['Main Location', 'Price(Cr)', 'Bath(s)', 'Area(Marla)', 'Bedroom(s)',
       'Description', 'Amenities: Main Features', 'Amenities: Rooms',
       'Amenities: Healthcare Recreational', 'Amenities: Other Facilities',
       'IsPrimeLoc', 'SolarInstalled', 'WaterBore', 'CornerHouse',
       'luxury_type', 'Built in year', 'Parking Spaces', 'Floor', 'Bedrooms',
       'Bathrooms', 'Servant Quarters', 'Kitchens', 'Store Rooms',
       'Swimming Pool', 'Storey Unit', 'BedBath_Diff'],
      dtype='str')

In [4]:
# First we remove the featuress we dont need

cols_to_remove = [
    
    'Parking Spaces','Amenities: Main Features', 'Amenities: Rooms',
    'Amenities: Healthcare Recreational',
    'Amenities: Other Facilities','Swimming Pool']

houses.drop(columns=cols_to_remove,inplace = True)

In [5]:
houses.info()

<class 'pandas.DataFrame'>
Index: 4383 entries, 0 to 6482
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Main Location     4383 non-null   str    
 1   Price(Cr)         4383 non-null   float64
 2   Bath(s)           4298 non-null   float64
 3   Area(Marla)       4383 non-null   float64
 4   Bedroom(s)        4308 non-null   float64
 5   Description       4383 non-null   str    
 6   IsPrimeLoc        4383 non-null   bool   
 7   SolarInstalled    4383 non-null   bool   
 8   WaterBore         4383 non-null   bool   
 9   CornerHouse       4383 non-null   bool   
 10  luxury_type       4383 non-null   int64  
 11  Built in year     3346 non-null   float64
 12  Floor             3256 non-null   float64
 13  Bedrooms          3796 non-null   float64
 14  Bathrooms         3793 non-null   float64
 15  Servant Quarters  2931 non-null   float64
 16  Kitchens          3524 non-null   float64
 17  Store Rooms

In [6]:
houses.isnull().sum()

Main Location          0
Price(Cr)              0
Bath(s)               85
Area(Marla)            0
Bedroom(s)            75
Description            0
IsPrimeLoc             0
SolarInstalled         0
WaterBore              0
CornerHouse            0
luxury_type            0
Built in year       1037
Floor               1127
Bedrooms             587
Bathrooms            590
Servant Quarters    1452
Kitchens             859
Store Rooms         1572
Storey Unit          752
BedBath_Diff          86
dtype: int64

- Bedroom(s) and Bath(s) --> Fill from Bathrooms and Bedrooms, then use Multivariate Approach
- Built in Year --> Fill from Location
- Storey Unit --> Fill from Floor, then use Multivariate approach
- Kitchens --> Can fill from Floor
- Store room & Servant Quarters --> Use Multivariate Approach

### Bedroom(s) & Bath(s)

In [7]:
houses[(houses['Bath(s)'].isnull())|(houses['Bedroom(s)'].isnull())][['Bathrooms','Bedrooms','Area(Marla)','Storey Unit','Floor','Bath(s)','Bedroom(s)']]

,Bathrooms,Bedrooms,Area(Marla),Storey Unit,Floor,Bath(s),Bedroom(s)
79,NaN,NaN,24.0,2.0,NaN,NaN,7.0
115,NaN,NaN,4.4,NaN,NaN,NaN,NaN
116,NaN,NaN,4.4,NaN,NaN,NaN,NaN
128,NaN,NaN,10.0,NaN,NaN,NaN,NaN
267,NaN,NaN,9.3,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...
5664,NaN,NaN,26.0,NaN,NaN,NaN,NaN
6030,NaN,NaN,15.0,1.0,NaN,NaN,NaN
6033,NaN,NaN,9.3,2.0,NaN,NaN,NaN
6065,NaN,NaN,7.0,2.0,NaN,NaN,NaN


In [8]:
houses[(houses['Bath(s)'].isnull()) & (houses['Bedroom(s)'].isnull())].shape

(74, 20)

*Removing the rows where both Bath(s) & Bedroom(s) are null*

In [9]:
houses.dropna(subset=['Bath(s)','Bedroom(s)'],inplace=True)

In [10]:
houses.shape

(4297, 20)

In [11]:
houses.isnull().sum()

Main Location          0
Price(Cr)              0
Bath(s)                0
Area(Marla)            0
Bedroom(s)             0
Description            0
IsPrimeLoc             0
SolarInstalled         0
WaterBore              0
CornerHouse            0
luxury_type            0
Built in year        957
Floor               1046
Bedrooms             502
Bathrooms            505
Servant Quarters    1368
Kitchens             775
Store Rooms         1486
Storey Unit          685
BedBath_Diff           0
dtype: int64

### Storey Unit

*How to impute values for Storey Unit?*
- First fill from floor
- If Floor is Nan, use Multivariate approach 

In [12]:
houses[(houses['Storey Unit'].isnull()) & ~(houses['Floor'].isnull())].shape

(0, 20)

*Cannot fill from Floor because where there is Nan value for Storey Unit, the corresponding value for Floor is also Nan <br> Using Multivariate Approach*

*Removing Bedrooms, Bathrooms, Floor, BedBath_Diff*

In [13]:
houses.drop(columns=['BedBath_Diff','Floor', 'Bedrooms',
       'Bathrooms'],inplace=True)

In [14]:
houses.isnull().sum()

Main Location          0
Price(Cr)              0
Bath(s)                0
Area(Marla)            0
Bedroom(s)             0
Description            0
IsPrimeLoc             0
SolarInstalled         0
WaterBore              0
CornerHouse            0
luxury_type            0
Built in year        957
Servant Quarters    1368
Kitchens             775
Store Rooms         1486
Storey Unit          685
dtype: int64

### Built in Year

1. First of all we are going to convert the Year column to Category.
2. Then fill the missing values based on Main Location


In [15]:
houses['Built in year'].value_counts()

Built in year
2025.0    1740
2024.0     529
2023.0     203
2022.0     114
2020.0     112
2015.0      82
2021.0      68
2018.0      61
2000.0      53
2010.0      52
2016.0      38
2026.0      37
2019.0      34
2017.0      31
2005.0      22
2012.0      22
2014.0      21
1990.0      17
1999.0      12
1998.0      12
2002.0      10
2013.0       9
2008.0       8
2009.0       8
2001.0       7
1995.0       5
2011.0       4
2027.0       3
1997.0       3
1985.0       3
2055.0       2
2007.0       2
1992.0       2
1988.0       2
2006.0       1
2003.0       1
1980.0       1
1978.0       1
1982.0       1
2029.0       1
2028.0       1
1981.0       1
1970.0       1
1984.0       1
1987.0       1
2004.0       1
Name: count, dtype: int64

In [16]:
houses.groupby('Built in year')['Price(Cr)'].mean().sort_index()

Built in year
1970.0     4.900000
1978.0    24.990000
1980.0    43.000000
1981.0    30.000000
1982.0    70.000000
1984.0    30.000000
1985.0    18.216667
1987.0     3.850000
1988.0    11.000000
1990.0    22.544118
1992.0    55.000000
1995.0    21.700000
1997.0    14.000000
1998.0    22.500000
1999.0    24.410000
2000.0    19.682642
2001.0    17.714286
2002.0    23.100000
2003.0    19.500000
2004.0     1.450000
2005.0    18.595455
2006.0    33.000000
2007.0     5.300000
2008.0    23.031250
2009.0    32.456250
2010.0    13.506731
2011.0    13.287500
2012.0    19.009091
2013.0     8.716667
2014.0    12.596667
2015.0    10.376707
2016.0     7.560000
2017.0     8.733226
2018.0     9.402951
2019.0     7.278824
2020.0    10.133036
2021.0     8.453088
2022.0    11.127544
2023.0    11.032842
2024.0     8.999168
2025.0     7.830756
2026.0     8.864324
2027.0     3.066667
2028.0     8.550000
2029.0     4.850000
2055.0     4.275000
Name: Price(Cr), dtype: float64

- Future --> $>2025$
- Immediate / Brand New --> $2025$
- Recently Built --> $2020 - 2024$
- Modern Era --> $2010 - 2019$
- Established --> $2000 - 2009$
- Vintage / Legacy-- > $< 2000$

In [17]:
# Converting 

bins = [-float('inf'),1999,2009,2019,2024,2025,float('inf')]
labels = ['Vintage','Established','Modern Era','Recently Built','Brand New (2025)','Future']

houses['Property era'] =  pd.cut(houses['Built in year'],bins=bins,labels=labels)

In [18]:
houses['Property era'].value_counts()

Property era
Brand New (2025)    1740
Recently Built      1026
Modern Era           354
Established          113
Vintage               63
Future                44
Name: count, dtype: int64

In [19]:
houses.groupby('Property era')['Price(Cr)'].mean().sort_index()

Property era
Vintage             23.651746
Established         20.493186
Modern Era          10.583729
Recently Built       9.725611
Brand New (2025)     7.830756
Future               8.162045
Name: Price(Cr), dtype: float64

In [20]:
houses.groupby('Property era')['Area(Marla)'].mean().sort_index()

Property era
Vintage             21.861905
Established         20.868142
Modern Era          14.251130
Recently Built      13.111111
Brand New (2025)    11.271552
Future              10.629545
Name: Area(Marla), dtype: float64

*Old properties have greater prices on average, than the new properties. One reason could be that old properties had greater land rather than the New ones.* 

In [21]:
houses[houses['Property era'].isnull()]['Main Location'].unique()

<ArrowStringArray>
[                          'D-12',                          'FECHS',
            'Faisal Town Phase 1',                           'D-17',
                            'G-9',                    'Ghauri Town',
                      'Bani Gala',                            'F-8',
                  'Margalla Town',                           'B-17',
                    'Mumtaz City',                    'DHA Phase 1',
                     'DHA Valley',                            'I-8',
                           'H-13',                    'DHA Phase 2',
                    'Bahria Town',                           'I-10',
             'Gulberg Residencia',                           'G-15',
                           'G-13',                            'F-7',
                 'Bahria Enclave',                    'DHA Phase 5',
                           'F-10',                           'F-11',
                     'Khanna Pul',                  'Jhangi Syedan',
          'MPCH

In [22]:
houses.groupby("Main Location")['Property era'].count().sort_values()

Main Location
G-12                0
Sihala              0
Thalian             0
PAF Tarnol          0
Park Road           0
                 ... 
Bani Gala         172
D-12              189
Bahria Enclave    210
G-13              259
DHA Phase 2       362
Name: Property era, Length: 113, dtype: int64

In [23]:
houses.head(1)

,Main Location,Price(Cr),Bath(s),Area(Marla),Bedroom(s),Description,IsPrimeLoc,SolarInstalled,WaterBore,CornerHouse,luxury_type,Built in year,Servant Quarters,Kitchens,Store Rooms,Storey Unit,Property era
0,D-12,12.0,7.0,10.0,6.0,10 MARLA LUXXARY BRAND NEW PARK FACE AVAILABLE...,False,False,False,False,0,2025.0,2.0,2.0,1.0,2.0,Brand New (2025)


In [24]:
def impute_propert_era(row):
    if pd.isna(row['Property era']):
        mode_location = houses[(houses['Main Location'] == row['Main Location'])]['Property era'].mode()

        if not mode_location.empty:
            return mode_location.iloc[0]
        else:
            return np.nan

    else:
        return row['Property era']
    


In [25]:
houses.apply(impute_propert_era,axis=1)

0       Brand New (2025)
1       Brand New (2025)
2       Brand New (2025)
3       Brand New (2025)
4       Brand New (2025)
              ...       
6472    Brand New (2025)
6478          Modern Era
6479          Modern Era
6480    Brand New (2025)
6482              Future
Length: 4297, dtype: str

In [26]:
location_modes =  houses.groupby('Main Location')['Property era'].apply(lambda x : x.mode().iloc[0] if not x.mode().empty else np.nan )

In [27]:
location_modes

Main Location
Airport Enclave         Brand New (2025)
Airport Green Garden    Brand New (2025)
Al Qaim Town              Recently Built
Alipur Farash           Brand New (2025)
Arsalan Town            Brand New (2025)
                              ...       
Thalian                              NaN
Thanda Pani                  Established
Top City 1              Brand New (2025)
University Town           Recently Built
Zaraj Housing Scheme          Modern Era
Name: Property era, Length: 113, dtype: str

In [28]:
houses['Main Location'].nunique()

113

In [29]:
houses['Main Location'].map(location_modes)

0       Brand New (2025)
1       Brand New (2025)
2       Brand New (2025)
3       Brand New (2025)
4       Brand New (2025)
              ...       
6472    Brand New (2025)
6478    Brand New (2025)
6479    Brand New (2025)
6480    Brand New (2025)
6482      Recently Built
Name: Main Location, Length: 4297, dtype: str

In [30]:
houses['Property era'] =  houses['Property era'].fillna(houses['Main Location'].map(location_modes))

In [31]:
houses['Property era'].isnull().sum()

np.int64(13)

In [32]:
# Filling the value with the mode now

houses['Property era'] = houses['Property era'].fillna(houses['Property era'].mode()[0])

In [33]:
# Removing Built in Year
houses.drop(columns='Built in year',inplace=True)

In [34]:
houses.isnull().sum()

Main Location          0
Price(Cr)              0
Bath(s)                0
Area(Marla)            0
Bedroom(s)             0
Description            0
IsPrimeLoc             0
SolarInstalled         0
WaterBore              0
CornerHouse            0
luxury_type            0
Servant Quarters    1368
Kitchens             775
Store Rooms         1486
Storey Unit          685
Property era           0
dtype: int64

In [35]:
# Saving the file for Streamlit purposes
houses.to_csv('houses.csv')

### Servant Quarters, Kitchens, Store Rooms, Storey Unit

Using Iterative Imputer(Tree based model) to fill the missing values because we haven't scaled our data.

Making a pipeline and doing a Randomized Search CV to find the best hyper parameters for imputing the values.

In [13]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer,SimpleImputer,KNNImputer

from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import RandomizedSearchCV,GridSearchCV

from sklearn import set_config

set_config(transform_output='pandas')

In [14]:
cols_to_impute = ['Bath(s)', 'Area(Marla)', 'Bedroom(s)','Servant Quarters', 'Kitchens', 'Store Rooms','Storey Unit']

In [15]:
trf_simple_imputer = ColumnTransformer(
    [
        ('SimpleImputer',SimpleImputer(),cols_to_impute)
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

trf_iter_imputer = ColumnTransformer(
    [
        ('IterImputer',IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=42)),cols_to_impute)
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

trf_knn_imputer = ColumnTransformer(
    [
        ('KnnImputer',KNNImputer(),cols_to_impute)
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

In [16]:
final_pipeline = Pipeline(
    [
        ('Impute',None),
        ('Model',XGBRegressor(n_estimators=100))
    ]
)

In [17]:
param_grid = [
    {
        "Impute":[trf_simple_imputer],
        'Impute__SimpleImputer__strategy':['mean','most_frequent']
    },
    {
        "Impute":[trf_iter_imputer],
        "Impute__IterImputer__max_iter":[10,20,30],
        "Impute__IterImputer__n_nearest_features": [3, 5, 10],
        "Impute__IterImputer__imputation_order":['random','ascending','arabic']
    },
    {
        "Impute":[trf_knn_imputer],
        "Impute__KnnImputer__n_neighbors":[10,20,30],
        "Impute__KnnImputer__weights":['distance',"uniform"]
    }
    
]



In [18]:
from sklearn.model_selection import train_test_split

X = houses[['Bath(s)', 'Area(Marla)', 'Bedroom(s)','Servant Quarters', 'Kitchens', 'Store Rooms','Storey Unit']]
y = houses['Price(Cr)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [30]:
search_cv = RandomizedSearchCV(final_pipeline,param_grid,n_iter=20,scoring='r2',n_jobs=-1,cv=5,)

search_cv.fit(X_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'Impute': [ColumnTransfo...mes_out=False)], 'Impute__SimpleImputer__strategy': ['mean', 'most_frequent']}, {'Impute': [ColumnTransfo...mes_out=False)], 'Impute__IterImputer__imputation_order': ['random', 'ascending', ...], 'Impute__IterImputer__max_iter': [10, 20, ...], 'Impute__IterImputer__n_nearest_features': [3, 5, ...]}, ...]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFo

In [31]:
search_cv.best_score_

np.float64(0.685942908573906)

In [32]:
search_cv.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Impute', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('KnnImputer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains

In [33]:
results = pd.DataFrame(search_cv.cv_results_)

In [34]:
results.sort_values(by=['mean_test_score'],ascending=False)[['mean_test_score', 'std_test_score','split0_test_score',
       'split1_test_score', 'split2_test_score', 'split3_test_score',
       'split4_test_score']]

,mean_test_score,std_test_score,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score
17,0.685943,0.022173,0.681021,0.659001,0.726746,0.680212,0.682735
7,0.678889,0.018105,0.702151,0.648312,0.684873,0.671609,0.687498
5,0.674898,0.040315,0.702299,0.616781,0.724384,0.639310,0.691717
0,0.673773,0.015536,0.667340,0.663595,0.701713,0.657691,0.678524
3,0.672985,0.008890,0.679290,0.672111,0.668455,0.659617,0.685452
6,0.672304,0.027352,0.690769,0.624399,0.703840,0.663902,0.678612
13,0.672173,0.019146,0.685389,0.653290,0.703131,0.656107,0.662949
10,0.671721,0.041892,0.625289,0.619716,0.706592,0.685093,0.721916
18,0.671161,0.024697,0.674590,0.642784,0.710181,0.646645,0.681604
11,0.670466,0.046378,0.636457,0.595872,0.704135,0.701125,0.714739


In [35]:
search_cv.best_params_

{'Impute__KnnImputer__weights': 'uniform',
 'Impute__KnnImputer__n_neighbors': 30,
 'Impute': ColumnTransformer(remainder='passthrough',
                   transformers=[('KnnImputer', KNNImputer(),
                                  ['Bath(s)', 'Area(Marla)', 'Bedroom(s)',
                                   'Servant Quarters', 'Kitchens', 'Store Rooms',
                                   'Storey Unit'])],
                   verbose_feature_names_out=False)}

In [19]:
trf_best_imputer = ColumnTransformer(
    [
        ('Iter',KNNImputer(weights='uniform',n_neighbors=30),cols_to_impute)
    ],
    remainder='passthrough',verbose_feature_names_out=False,n_jobs=-1
)

best_pipeline =Pipeline(
    [
        ('Impute',trf_best_imputer),
        ('Model',XGBRegressor(n_estimators=100,n_jobs=-1))
    ]
)

In [21]:
best_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('Impute', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Iter', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spars

In [22]:
from sklearn.metrics import r2_score,mean_absolute_error

y_pred = best_pipeline.predict(X_test)
print('R2 Score',r2_score(y_test,y_pred))
print('Mean Absolute Error',mean_absolute_error(y_test,y_pred))

R2 Score 0.7256260506845067
Mean Absolute Error 2.9700130531611872


In [38]:
# Using KNN Imputer to Fill the Missing Values

In [107]:
houses.columns

Index(['Main Location', 'Price(Cr)', 'Bath(s)', 'Area(Marla)', 'Bedroom(s)',
       'Description', 'IsPrimeLoc', 'SolarInstalled', 'WaterBore',
       'CornerHouse', 'luxury_type', 'Servant Quarters', 'Kitchens',
       'Store Rooms', 'Storey Unit', 'Property era'],
      dtype='str')

In [131]:
from sklearn.model_selection import train_test_split

X = houses[['Main Location', 'Bath(s)', 'Area(Marla)', 'Bedroom(s)',
        'IsPrimeLoc', 'SolarInstalled', 'WaterBore',
       'CornerHouse', 'luxury_type', 'Servant Quarters', 'Kitchens',
       'Store Rooms', 'Storey Unit', 'Property era']]
y = houses['Price(Cr)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [132]:
X_train = round(trf_best_imputer.fit_transform(X_train))

X_test = round(trf_best_imputer.transform(X_test))

In [133]:
X_train.shape

(2878, 14)

In [134]:
X_train.isnull().sum()

Bath(s)             0
Area(Marla)         0
Bedroom(s)          0
Servant Quarters    0
Kitchens            0
Store Rooms         0
Storey Unit         0
Main Location       0
IsPrimeLoc          0
SolarInstalled      0
WaterBore           0
CornerHouse         0
luxury_type         0
Property era        0
dtype: int64

In [136]:
X_test.isnull().sum()

Bath(s)             0
Area(Marla)         0
Bedroom(s)          0
Servant Quarters    0
Kitchens            0
Store Rooms         0
Storey Unit         0
Main Location       0
IsPrimeLoc          0
SolarInstalled      0
WaterBore           0
CornerHouse         0
luxury_type         0
Property era        0
dtype: int64

In [137]:
# Saving X_train, X_test, y_train, y_test

In [138]:
X_train.to_csv('Houses/X_train.csv')
X_test.to_csv('Houses/X_test.csv')
y_train.to_csv('Houses/y_train.csv')
y_test.to_csv('Houses/y_test.csv')